<a href="https://colab.research.google.com/github/Jorge-Ruiz-Troccoli/Data-Science-I/blob/main/Clase%2002/RedesNeuronales_vs_RegresionLineal_PreciosCasas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏠 Precio de Casas: Regresión Lineal vs Red Neuronal
Clase 2 — Data Science | Coderhouse 🚀

**Objetivo:** comparar un modelo clásico (Regresión Lineal) contra una Red Neuronal en el mismo problema, y ver si realmente vale la pena la complejidad extra.

## 📥 Carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras


In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame
df.head()

## ✂️ Train / Test split + escalado
Ojo: el `scaler` se ajusta **solo con train** (lección de la clase pasada 👀), nunca con todo el dataset.

In [ ]:
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
x_train_s = scaler.fit_transform(x_train)
x_test_s  = scaler.transform(x_test)

## 📏 Modelo 1: Regresión Lineal (baseline clásico)

In [ ]:
lin_model = LinearRegression()
lin_model.fit(x_train_s, y_train)

y_pred_lin = lin_model.predict(x_test_s)
rmse_lin = mean_squared_error(y_test, y_pred_lin) ** 0.5
r2_lin = r2_score(y_test, y_pred_lin)
print("RMSE:", rmse_lin)
print("R2  :", r2_lin)

## 🧠 Modelo 2: Red Neuronal (mismo problema, mismos datos)

In [ ]:
def define_model():
    keras.backend.clear_session()
    model = keras.models.Sequential()
    model.add(keras.layers.Dense(16, input_dim=x_train_s.shape[1], activation='relu'))
    model.add(keras.layers.Dense(8, activation='relu'))
    model.add(keras.layers.Dense(1))  # regresión: sin activación en la salida
    model.compile(loss='mse', metrics=['mae'], optimizer='adam')
    return model

nn_model = define_model()
early = keras.callbacks.EarlyStopping(patience=10, monitor='val_loss', restore_best_weights=True)
history = nn_model.fit(x_train_s, y_train, validation_data=(x_test_s, y_test),
                        epochs=100, batch_size=64, callbacks=[early], verbose=0)

In [ ]:
y_pred_nn = nn_model.predict(x_test_s).flatten()
rmse_nn = mean_squared_error(y_test, y_pred_nn) ** 0.5
r2_nn = r2_score(y_test, y_pred_nn)
print("RMSE:", rmse_nn)
print("R2  :", r2_nn)

## ⚖️ Comparación: ¿da lo mismo?

In [ ]:
comparacion = pd.DataFrame({
    'Modelo': ['Regresión Lineal', 'Red Neuronal'],
    'RMSE': [rmse_lin, rmse_nn],
    'R2': [r2_lin, r2_nn]
})
comparacion

In [ ]:
plt.bar(comparacion['Modelo'], comparacion['RMSE'], color=['#4C72B0', '#DD8452'])
plt.ylabel('RMSE (menor es mejor)')
plt.title('Regresión Lineal vs Red Neuronal')
plt.show()

## 🤝 Conclusión
Cuando la relación entre features y target es **mayormente lineal**, una Red Neuronal no mejora mucho a una Regresión Lineal — pero consume muchísimo más tiempo de entrenamiento y cómputo. Las redes neuronales brillan cuando hay **relaciones no lineales complejas** que un modelo lineal no puede capturar. Moraleja: probá siempre el modelo simple primero. 💡